# NFHS-5: How many women aged 30–49 have ever been screened for cancer?

Survey-weighted percentages of women aged 30–49 who have ever been screened for **cervical**, **breast** and **oral** cancer,
overall and by place of residence, wealth and education.

**Data rule (DHS terms):** the `.DTA` and `.pkl` files stay on this laptop. Never upload them anywhere.
Only the printed table and the saved CSV (summary numbers) may be shared.

In [ ]:
# SETTINGS: run this first
import os, time
import numpy as np
import pandas as pd

FOLDER  = r"D:\DHS dataset"                                     # your data folder (stays on your laptop)
DTA     = os.path.join(FOLDER, "IAIR7EFL.DTA")                  # NFHS-5 women's file
SUBSET1 = os.path.join(FOLDER, "nfhs5_screening_subset.pkl")    # small file from Step 1 (made below if missing)
RESULTS = os.path.join(FOLDER, "results")                       # summary tables only
os.makedirs(RESULTS, exist_ok=True)
pd.set_option("display.width", 200)
print("Settings ready.")

## Cell 1: Load the small file (or create it)
If the Step 1 file already exists, it is loaded in seconds. If not, the needed variables are read from the big file
in pieces of 100,000 women (a few minutes). Then we keep women aged 30–49.

In [ ]:
# CELL 1: load or create the small file, keep ages 30-49
cols = ["v005", "v012", "v025", "v106", "v190", "sb79", "sb80", "sb81"]
if os.path.exists(SUBSET1):
    df = pd.read_pickle(SUBSET1)
    print("Loaded the existing small file.")
else:
    t0 = time.time()
    parts = [chunk for chunk in pd.read_stata(DTA, columns=cols, convert_categoricals=False, chunksize=100_000)]
    df = pd.concat(parts, ignore_index=True)
    df.to_pickle(SUBSET1)
    print(f"Created the small file ({time.time() - t0:.0f} seconds).")

d = df[(df["v012"] >= 30) & (df["v012"] <= 49)].copy()
d["w"] = d["v005"] / 1_000_000          # survey weight
print(f"Women aged 30-49: {len(d):,}")

## Cell 2: Weighted percentages by group
For each outcome, women with no answer are left out, and each woman counts by her survey weight (`v005`).

In [ ]:
# CELL 2: weighted % ever screened, by group
groups = {
    "v025": {1: "Urban", 2: "Rural"},
    "v190": {1: "Poorest", 2: "Poorer", 3: "Middle", 4: "Richer", 5: "Richest"},
    "v106": {0: "No education", 1: "Primary", 2: "Secondary", 3: "Higher"},
}
outcomes = {"sb79": "Cervical %", "sb80": "Breast %", "sb81": "Oral %"}

def weighted_pct(g, col):
    answered = g[col].notna()
    return 100 * np.average(g.loc[answered, col] == 1, weights=g.loc[answered, "w"])

rows = [{"Variable": "all", "Group": "All women", "Women": len(d),
         **{name: weighted_pct(d, col) for col, name in outcomes.items()}}]
for var, names in groups.items():
    for code_, name in names.items():
        g = d[d[var] == code_]
        rows.append({"Variable": var, "Group": name, "Women": len(g),
                     **{n: weighted_pct(g, c) for c, n in outcomes.items()}})

table = pd.DataFrame(rows)
print(table.round(2).to_string(index=False))
table.round(3).to_csv(os.path.join(RESULTS, "screening_rates_by_group.csv"), index=False)

rich = table.loc[table["Group"] == "Richest", "Cervical %"].iloc[0]
poor = table.loc[table["Group"] == "Poorest", "Cervical %"].iloc[0]
print(f"\nCervical screening, richest vs poorest: {rich:.2f}% vs {poor:.2f}% = {rich / poor:.1f} times")